# Notebook 03: Dataset Generation

## Objective

In this notebook, we will:

- Process all ASL images using MediaPipe.
- Extract 21 hand landmarks (63 features).
- Assign labels to each sample.
- Create a structured dataset.
- Save the dataset as `landmarks.csv` for machine learning model training.

**Output:**

- dataset/processed/landmarks.csv

In [32]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm
import time

In [34]:
PROJECT_ROOT = Path("..")

TRAIN_PATH = PROJECT_ROOT / "dataset" / "raw" / "asl_alphabet_train"

PROCESSED_PATH = PROJECT_ROOT / "dataset" / "processed"

PROCESSED_PATH.mkdir(exist_ok=True)

#MediaPipe Initialization
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.30,
    min_tracking_confidence=0.30
)

In [3]:
classes = sorted(
    [
        folder.name
        for folder in TRAIN_PATH.iterdir()
        if folder.is_dir()
    ]
)

print(classes)
print(f"\nTotal Classes: {len(classes)}")

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']

Total Classes: 29


In [4]:
columns = []

for i in range(21):
    columns.extend([
        f"x{i}",
        f"y{i}",
        f"z{i}"
    ])

columns.append("label")

print("Total Columns:", len(columns))

Total Columns: 64


In [25]:
def brighten(image):
    """
    Increase image brightness.
    """
    return cv2.convertScaleAbs(image, alpha=1.3, beta=30)


def gamma(image, gamma=1.5):
    """
    Apply gamma correction.
    """
    inv_gamma = 1.0 / gamma

    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255
        for i in range(256)
    ]).astype("uint8")

    return cv2.LUT(image, table)

In [26]:
#Test Image
def read_image(image_path):
    """
    Reads an image from disk and converts it to RGB.
    """

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    return image_rgb

sample_image = next((TRAIN_PATH / "A").glob("*"))

img = read_image(sample_image)

print(img.shape)

(200, 200, 3)


In [27]:
#Extract Landmarks
def extract_landmarks(image_rgb):
    """
    Extract 63 landmark features.
    """

    results = hands.process(image_rgb)

    if not results.multi_hand_landmarks:
        return None

    hand_landmarks = results.multi_hand_landmarks[0]

    features = []

    for landmark in hand_landmarks.landmark:
        features.extend([
            landmark.x,
            landmark.y,
            landmark.z
        ])

    if len(features) != 63:
        return None

    return features

In [37]:
#Processing image
def process_image(image_path):
    """
    Read an image and extract landmarks using
    multiple enhancement strategies.

    Returns:
        features, method_used
    """

    image = cv2.imread(str(image_path))

    if image is None:
        return None, None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # -----------------------------
    # Attempt 1 : Original
    # -----------------------------

    features = extract_landmarks(image_rgb)

    if features is not None:
        return features, "original"

    # -----------------------------
    # Attempt 2 : Brightness
    # -----------------------------

    bright = brighten(image_rgb)

    features = extract_landmarks(bright)

    if features is not None:
        return features, "brightness"

    # -----------------------------
    # Attempt 3 : Gamma
    # -----------------------------

    gamma_img = gamma(image_rgb)

    features = extract_landmarks(gamma_img)

    if features is not None:
        return features, "gamma"

    return None, None

In [38]:
#Test on 1 image
sample_image = next((TRAIN_PATH / classes[0]).glob("*"))

features, method = process_image(sample_image)

print(method)
print(len(features))

original
63


d:\Code FIles\VS code files\Project\sign-language-translator\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [41]:
#Testing few images
count = 0

for image_path in (TRAIN_PATH / "A").glob("*"):

    features, method = process_image(image_path)

    if features is not None:
        print(f"Method: {method}, Features: {len(features)}")

    count += 1

    if count == 5:
        break

Method: original, Features: 63
Method: original, Features: 63
Method: brightness, Features: 63
Method: original, Features: 63
Method: original, Features: 63


In [42]:
# Start timer
start = time.time()

dataset = []
failed_images = []

method_count = {
    "original": 0,
    "brightness": 0,
    "gamma": 0
}

for class_name in tqdm(classes, desc="Processing Classes"):

    class_folder = TRAIN_PATH / class_name

    image_paths = list(class_folder.glob("*"))

    for image_path in image_paths:

        features, method = process_image(image_path)

        if features is None:
            failed_images.append(str(image_path))
            continue

        method_count[method] += 1

        features.append(class_name)

        dataset.append(features)


# Stop timer
end = time.time()

print("=" * 50)
print(f"Successful Samples : {len(dataset)}")
print(f"Failed Samples     : {len(failed_images)}")
print("\nDetection Statistics")
print("-" * 50)

for method, count in method_count.items():
    print(f"{method:<12}: {count}")

Processing Classes: 100%|██████████| 29/29 [1:39:49<00:00, 206.52s/it]

Successful Samples : 74137
Failed Samples     : 12863

Detection Statistics
--------------------------------------------------
original    : 66995
brightness  : 4732
gamma       : 2410


In [45]:
import pandas as pd

failure_stats = []

for cls in classes:
    total = len(list((TRAIN_PATH / cls).glob("*")))
    failed = sum(1 for img in failed_images if Path(img).parent.name == cls)

    failure_stats.append({
        "Class": cls,
        "Total": total,
        "Failed": failed,
        "Failure %": round(failed / total * 100, 2)
    })

failure_df = pd.DataFrame(failure_stats)

failure_df.sort_values("Failure %", ascending=False)

,Class,Total,Failed,Failure %
27,nothing,3000,2964,98.80
13,N,3000,1056,35.20
12,M,3000,721,24.03
26,del,3000,713,23.77
23,X,3000,643,21.43
15,P,3000,526,17.53
28,space,3000,511,17.03
4,E,3000,478,15.93
16,Q,3000,472,15.73
19,T,3000,399,13.30


In [50]:
df = pd.DataFrame(dataset, columns=columns)

In [51]:
print(df.shape)

(74137, 64)


In [52]:
df.isnull().sum().sum()

np.int64(0)

In [53]:
df["label"].value_counts()

label
F          2998
J          2912
D          2903
L          2849
H          2844
G          2832
K          2820
U          2774
R          2771
B          2770
S          2765
V          2759
Y          2720
W          2681
Z          2673
I          2668
O          2642
C          2622
A          2617
T          2601
Q          2528
E          2522
space      2489
P          2474
X          2357
del        2287
M          2279
N          1944
nothing      36
Name: count, dtype: int64

In [57]:
csv_path = PROCESSED_PATH / "landmarks.csv"

df.to_csv(csv_path, index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [58]:
import os

print(os.listdir(PROCESSED_PATH))

['checkpoints', 'landmarks.csv']
